# Retinal Disease Detection with Vision Transformer (ViT)

**Fine-tune a ViT model to detect 8 eye diseases from fundus photographs.**

Diseases detected:
- **Normal** — No abnormalities
- **Diabetic Retinopathy** — Damage to retinal blood vessels from diabetes
- **Glaucoma** — Optic nerve damage from increased eye pressure
- **Cataract** — Clouding of the eye's lens
- **AMD** — Age-related macular degeneration
- **Hypertension** — Retinal damage from high blood pressure
- **Myopia** — Pathological nearsightedness
- **Other** — Other abnormalities

**Dataset:** ODIR-5K (Ocular Disease Intelligent Recognition)  
**Model:** google/vit-base-patch16-224  
**Runtime:** ~1 hour on T4 GPU (Colab free tier)

## 1. Setup & Install Dependencies

In [ ]:
!pip install -q transformers datasets torch torchvision Pillow scikit-learn \
    matplotlib seaborn pandas kagglehub huggingface-hub accelerate openpyxl tqdm

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms
from transformers import ViTForImageClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Download ODIR-5K Dataset

We use `kagglehub` to download the dataset. You'll need a Kaggle account — get your API key from https://www.kaggle.com/settings and set it below.

In [ ]:
# Set your Kaggle credentials (or upload kaggle.json to ~/.kaggle/)
os.environ["KAGGLE_USERNAME"] = "eliasteikari"
os.environ["KAGGLE_KEY"] = 

import kagglehub

# Download ODIR-5K dataset
data_path = kagglehub.dataset_download("andrewmvd/ocular-disease-recognition-odir5k")
print(f"Dataset downloaded to: {data_path}")

# List contents
for item in os.listdir(data_path):
    full = os.path.join(data_path, item)
    size = os.path.getsize(full) if os.path.isfile(full) else "dir"
    print(f"  {item} ({size})")

## 3. Load & Explore the Dataset

In [ ]:
# Constants
DISEASE_CLASSES = ["Normal", "Diabetes", "Glaucoma", "Cataract", "AMD", "Hypertension", "Myopia", "Other"]
DISEASE_CODES = ["N", "D", "G", "C", "A", "H", "M", "O"]
NUM_CLASSES = len(DISEASE_CLASSES)

# Find annotation file
anno_file = None
for f in sorted(os.listdir(data_path)):
    if f.endswith(".xlsx") or f.endswith(".csv"):
        anno_file = os.path.join(data_path, f)
        print(f"Found annotation file: {f}")

# Load annotations
if anno_file.endswith(".xlsx"):
    df_raw = pd.read_excel(anno_file)
else:
    df_raw = pd.read_csv(anno_file)

print(f"\nAnnotation shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head()

In [ ]:
# Find image directory
img_dir = None
for d in os.listdir(data_path):
    full = os.path.join(data_path, d)
    if os.path.isdir(full):
        img_dir = full
        n_files = len(os.listdir(full))
        print(f"Found image directory: {d} ({n_files} files)")

if img_dir is None:
    img_dir = data_path
    print("Using data_path as image directory")

# Parse labels — assign each patient-eye the primary diagnosis
def parse_label(row):
    """Return the first positive disease class (0-7)."""
    for i, code in enumerate(DISEASE_CODES):
        if code in df_raw.columns and row[code] == 1:
            return i
    return 0  # Normal

# Build per-image records
records = []
missing = 0

for _, row in df_raw.iterrows():
    label = parse_label(row)

    for eye_col in ["Left-Fundus", "Right-Fundus"]:
        if eye_col not in row or pd.isna(row[eye_col]):
            continue
        img_path = os.path.join(img_dir, str(row[eye_col]))
        if os.path.exists(img_path):
            records.append({"image_path": img_path, "label": label, "label_name": DISEASE_CLASSES[label]})
        else:
            missing += 1

df = pd.DataFrame(records)
print(f"\nLoaded {len(df)} images ({missing} missing files skipped)")
print(f"\nClass distribution:")
print(df["label_name"].value_counts())

In [ ]:
# Visualize class distribution
fig, ax = plt.subplots(figsize=(10, 5))
counts = df["label_name"].value_counts()
colors = sns.color_palette("husl", NUM_CLASSES)
counts.plot(kind="bar", ax=ax, color=colors)
ax.set_title("ODIR-5K Class Distribution", fontsize=14)
ax.set_ylabel("Number of Images")
ax.set_xlabel("Disease Category")
plt.xticks(rotation=45, ha="right")
for i, v in enumerate(counts.values):
    ax.text(i, v + 20, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Show sample images from each class
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, (cls_name, ax) in enumerate(zip(DISEASE_CLASSES, axes.flat)):
    subset = df[df["label_name"] == cls_name]
    if len(subset) > 0:
        sample = subset.sample(1).iloc[0]
        img = Image.open(sample["image_path"]).convert("RGB")
        ax.imshow(img)
        ax.set_title(cls_name, fontsize=12, fontweight="bold")
    ax.axis("off")
plt.suptitle("Sample Fundus Images per Disease Category", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Prepare Data Loaders

In [ ]:
# Hyperparameters
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
LR = 2e-5
WEIGHT_DECAY = 0.01
VAL_SPLIT = 0.15
TEST_SPLIT = 0.10
SEED = 42

# Transforms
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
class RetinalDiseaseDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        label = row["label"]
        if self.transform:
            image = self.transform(image)
        return image, label

# Stratified train/val/test split
train_df, temp_df = train_test_split(df, test_size=VAL_SPLIT + TEST_SPLIT, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=TEST_SPLIT / (VAL_SPLIT + TEST_SPLIT), stratify=temp_df["label"], random_state=SEED)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Datasets
train_dataset = RetinalDiseaseDataset(train_df, transform=train_transforms)
val_dataset = RetinalDiseaseDataset(val_df, transform=val_transforms)
test_dataset = RetinalDiseaseDataset(test_df, transform=val_transforms)

# Class weights for imbalanced data
train_labels = train_df["label"].values
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES).astype(np.float32)
class_counts = np.maximum(class_counts, 1.0)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
class_weights = torch.FloatTensor(class_weights).to(device)
print(f"\nClass weights: {dict(zip(DISEASE_CLASSES, class_weights.cpu().numpy().round(2)))}")

# Weighted sampler for balanced batches
sample_weights = 1.0 / class_counts[train_labels]
sampler = WeightedRandomSampler(torch.DoubleTensor(sample_weights), len(sample_weights), replacement=True)

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

## 5. Load Pretrained ViT & Configure for Fine-Tuning

In [ ]:
# Load pretrained ViT with new classification head
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")

In [ ]:
# Loss, optimizer, scheduler
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.01)

## 6. Training Loop

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images).logits
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total, correct / total

In [ ]:
# Training
best_val_acc = 0.0
patience = 5
no_improve = 0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

OUTPUT_DIR = "retinal_disease_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Training for {EPOCHS} epochs...\n")

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch+1}/{EPOCHS} — "
        f"Train Loss: {train_loss:.4f}, Train Acc: {100*train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {100*val_acc:.2f}%"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        no_improve = 0
        model.save_pretrained(os.path.join(OUTPUT_DIR, "best_model"))
        print(f"  -> New best model saved! (val_acc: {100*val_acc:.2f}%)")
    else:
        no_improve += 1

    if no_improve >= patience:
        print(f"\nEarly stopping after {patience} epochs without improvement.")
        break

# Save history and label map
with open(os.path.join(OUTPUT_DIR, "history.json"), "w") as f:
    json.dump(history, f, indent=2)

label_map = {i: name for i, name in enumerate(DISEASE_CLASSES)}
with open(os.path.join(OUTPUT_DIR, "best_model", "label_map.json"), "w") as f:
    json.dump(label_map, f, indent=2)

print(f"\nTraining complete! Best validation accuracy: {100*best_val_acc:.2f}%")

## 7. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history["train_loss"]) + 1)

ax1.plot(epochs_range, history["train_loss"], "b-o", label="Train", markersize=4)
ax1.plot(epochs_range, history["val_loss"], "r-o", label="Validation", markersize=4)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training & Validation Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, [a*100 for a in history["train_acc"]], "b-o", label="Train", markersize=4)
ax2.plot(epochs_range, [a*100 for a in history["val_acc"]], "r-o", label="Validation", markersize=4)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Training & Validation Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
plt.show()

## 8. Evaluate on Test Set

In [ ]:
# Load best model
best_model = ViTForImageClassification.from_pretrained(os.path.join(OUTPUT_DIR, "best_model"))
best_model = best_model.to(device)
best_model.eval()

# Get all predictions
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(device)
        outputs = best_model(images).logits
        probs = torch.softmax(outputs, dim=1)
        all_preds.append(outputs.argmax(dim=1).cpu().numpy())
        all_labels.append(labels.numpy())
        all_probs.append(probs.cpu().numpy())

y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_labels)
y_probs = np.concatenate(all_probs)

# Classification report
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=DISEASE_CLASSES, digits=4))

# Overall accuracy
accuracy = (y_pred == y_true).mean()
print(f"Overall Test Accuracy: {100*accuracy:.2f}%")

In [ ]:
# Per-class AUC
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
print("Per-class AUC (One-vs-Rest):")
auc_scores = {}
for i, name in enumerate(DISEASE_CLASSES):
    if y_true_bin[:, i].sum() > 0:
        auc = roc_auc_score(y_true_bin[:, i], y_probs[:, i])
        auc_scores[name] = auc
        print(f"  {name:15s}: {auc:.4f}")

mean_auc = np.mean(list(auc_scores.values()))
print(f"  {'Mean AUC':15s}: {mean_auc:.4f}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=DISEASE_CLASSES, yticklabels=DISEASE_CLASSES)
plt.xlabel("Predicted", fontsize=12)
plt.ylabel("True", fontsize=12)
plt.title("Confusion Matrix — Retinal Disease Classification", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

## 9. Try It Out — Predict on Sample Images

In [ ]:
def predict_and_show(image_path, model, device):
    """Predict disease and display the image with results."""
    image = Image.open(image_path).convert("RGB")
    tensor = val_transforms(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(tensor).logits
        probs = torch.softmax(outputs, dim=1).squeeze().cpu().numpy()

    predicted_idx = probs.argmax()
    predicted_class = DISEASE_CLASSES[predicted_idx]
    confidence = probs[predicted_idx]

    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.imshow(image)
    ax1.set_title(f"Prediction: {predicted_class} ({confidence*100:.1f}%)", fontsize=14, fontweight="bold")
    ax1.axis("off")

    colors = ["green" if i == predicted_idx else "steelblue" for i in range(NUM_CLASSES)]
    bars = ax2.barh(DISEASE_CLASSES, probs * 100, color=colors)
    ax2.set_xlabel("Confidence (%)")
    ax2.set_title("Disease Probabilities")
    ax2.set_xlim(0, 100)
    for bar, prob in zip(bars, probs):
        if prob > 0.02:
            ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                     f"{prob*100:.1f}%", va="center", fontsize=10)

    plt.tight_layout()
    plt.show()

# Test on random samples from the test set
test_samples = test_df.sample(4, random_state=42)
for _, row in test_samples.iterrows():
    print(f"\nTrue label: {row['label_name']}")
    predict_and_show(row["image_path"], best_model, device)

## 10. Save & Export Model

The model is saved locally. You can optionally push it to HuggingFace Hub.

In [ ]:
# Download the model from Colab
print(f"Model saved at: {os.path.join(OUTPUT_DIR, 'best_model')}")
print(f"\nFiles:")
model_dir = os.path.join(OUTPUT_DIR, "best_model")
for f in os.listdir(model_dir):
    size_mb = os.path.getsize(os.path.join(model_dir, f)) / 1e6
    print(f"  {f} ({size_mb:.1f} MB)")

# Zip for easy download
!cd {OUTPUT_DIR} && zip -r best_model.zip best_model/
print(f"\nZipped model: {os.path.join(OUTPUT_DIR, 'best_model.zip')}")

In [ ]:
# Optional: Push to HuggingFace Hub
# Uncomment and fill in your details:

# from huggingface_hub import HfApi, login
# login()  # Will prompt for your HF token
#
# best_model.push_to_hub("your-username/retinal-disease-vit")
# print("Model pushed to HuggingFace Hub!")

## Done!

You now have a trained retinal disease detection model. Next steps:

1. **Download** the `best_model.zip` from Colab
2. **Run the Gradio app** locally: `python app/gradio_app.py --model_dir checkpoints/best_model`
3. **Deploy** to HuggingFace Spaces for a free hosted demo
4. **Show your dad!** Upload fundus images and get instant predictions

**Disclaimer:** This is a screening/educational tool, not a diagnostic device. Always consult a qualified ophthalmologist.